# Corpus y extracción: documentos y audios peruanos

- Recolección del piloto: 23 de octubre de 2026, UTC-5.
- Procedencia, licencias, atribución y acceso por archivo: Wikimedia Commons; registro en datos/fuentes.csv.
- PaddleOCR y ASR: todavía sin instalación ni inferencia; versiones y configuraciones se documentarán al preparar los entornos.
- Entornos de las sesiones: uv sync --group docai y uv sync --group audio.

| ID | Tipo | Autor o digitalización | Licencia |
|---|---|---|---|
| doc_001 | impreso | Helder Zavaleta Verde | [CC BY 4.0](https://commons.wikimedia.org/wiki/File:Acta_de_Creaci%C3%B3n_de_los_S%C3%ADmbolos_para_el_Distrito_de_Mache.jpg) |
| doc_002 | foto | Miguel Angel Chong | [CC BY 3.0](https://commons.wikimedia.org/wiki/File:Placa_Calle_Capon.jpg) |
| doc_003 | manuscrito | Leguicall | [Public domain](https://commons.wikimedia.org/wiki/File:Tarjeta_de_Ricardo_Palma_a_Luis_Antonio_Eguiguren_Escudero_24_octubre_1912.png) |
| clip_001 | entrevista | JorgeFlores-pe | [CC BY-SA 4.0](https://commons.wikimedia.org/wiki/File:Entrevista_2_en_la_Av._Rivag%C3%BCero_-_El_Agustino.ogg) |

La tabla documenta los archivos descargados del piloto, incluido el descartado; la selección se detalla en el apartado 1.2.


## 1. Configuración e inventario

Registrar versiones, rutas y parámetros. Reunir al menos 25 documentos peruanos: 8 impresos nítidos, 8 fotos con sombras o inclinación y 9 manuscritos; además, al menos 20 clips en español peruano que sumen 30 minutos, incluido uno con dos o más hablantes. Registrar fuente, licencia, atribución y acceso por archivo; documentar consentimiento para grabaciones propias. En descargas web, respetar robots.txt, usar un User-Agent descriptivo y pausas de 1 a 2 segundos.


In [1]:
# Metas de recolección; no representan archivos ya recolectados.
META_DOCUMENTOS = {'impreso': 8, 'foto': 8, 'manuscrito': 9}
MIN_CLIPS = 20
MIN_AUDIO_SEG = 30 * 60
MIN_AUDIOS_MULTIHABLANTE = 1

print('Meta de documentos:', META_DOCUMENTOS)
print('Total documental:', sum(META_DOCUMENTOS.values()))
print('Mínimo de clips:', MIN_CLIPS)
print('Mínimo de audio (minutos):', MIN_AUDIO_SEG / 60)
print('Mínimo de audios con dos o más hablantes:', MIN_AUDIOS_MULTIHABLANTE)


Meta de documentos: {'impreso': 8, 'foto': 8, 'manuscrito': 9}
Total documental: 25
Mínimo de clips: 20
Mínimo de audio (minutos): 30.0
Mínimo de audios con dos o más hablantes: 1


### 1.1. Selección inicial y procedencia

La selección prioriza procedencia verificable y variedad, sin un tema único. Esta prueba descarga tres documentos de distinto tipo y una entrevista usados en los materiales de clase. La licencia, el autor, el enlace original, la fecha de descarga y el hash se registran por archivo. Los originales y respaldos de sus fichas permanecen en `datos/raw/`, fuera de Git. La aceptación en el corpus requiere revisar también el contenido y su utilidad; tener licencia no equivale a haber revisado los datos personales.

Se consultan las páginas de archivo permitidas por robots.txt, con un User-Agent descriptivo y pausas de 1,5 segundos. No se usa la ruta Special:FilePath, excluida por robots.txt. Las descargas existentes se reutilizan.


In [2]:
from pathlib import Path
from datetime import datetime, timedelta, timezone
from html.parser import HTMLParser
from html import unescape
from urllib.parse import quote, urlsplit, urlunsplit
from urllib.error import HTTPError
from urllib.robotparser import RobotFileParser
import csv
import hashlib
import json
import re
import time
import urllib.request

RAIZ = Path.cwd()
if not (RAIZ / "datos").is_dir() and (RAIZ / "tarea_2" / "datos").is_dir():
    RAIZ = RAIZ / "tarea_2"
if not (RAIZ / "datos").is_dir():
    raise RuntimeError("Ejecutar desde tarea_2 o desde la raíz del repositorio.")
RAW = RAIZ / "datos" / "raw"
RAW.mkdir(parents=True, exist_ok=True)
AGENTE = "Tarea2Corpus/1.0 (academic project; https://github.com/bocchienjoyer/python_intermedio)"
robots_por_host = {}
ultima_peticion = 0.0

def peticion_http(url):
    global ultima_peticion
    time.sleep(max(0, 1.5 - (time.monotonic() - ultima_peticion)))
    solicitud = urllib.request.Request(url, headers={"User-Agent": AGENTE})
    try:
        with urllib.request.urlopen(solicitud, timeout=30) as respuesta:
            contenido = respuesta.read()
    finally:
        ultima_peticion = time.monotonic()
    return contenido

def obtener(url):
    partes = urlsplit(url)
    host = f"{partes.scheme}://{partes.netloc}"
    if host not in robots_por_host:
        robot = RobotFileParser()
        try:
            texto = peticion_http(host + "/robots.txt").decode("utf-8-sig")
            robot.parse(texto.splitlines())
            codigo = 200
        except HTTPError as error:
            if error.code != 404:
                raise
            texto, codigo = "", 404
            robot.parse([])
        robots_por_host[host] = (robot, {"url": host + "/robots.txt", "http": codigo, "contenido": texto})
    robot, _ = robots_por_host[host]
    if not robot.can_fetch(AGENTE, url):
        raise PermissionError(f"Ruta excluida por robots.txt: {url}")
    return peticion_http(url)

class EnlacesYTexto(HTMLParser):
    def __init__(self):
        super().__init__()
        self.enlaces, self.textos = [], []
    def handle_starttag(self, tag, attrs):
        attrs = dict(attrs)
        if tag == "a" and attrs.get("href"):
            self.enlaces.append(attrs["href"])
    def handle_data(self, dato):
        self.textos.append(dato)

def texto_sin_etiquetas(fragmento):
    pagina = EnlacesYTexto()
    pagina.feed(fragmento)
    return " ".join(" ".join(pagina.textos).split())

def metadatos_commons(html):
    html = unescape(html)
    pagina = EnlacesYTexto()
    pagina.feed(html)
    licencias = re.findall(r'<span[^>]*class="[^"]*licensetpl_short[^"]*"[^>]*>(.*?)</span>', html, re.S)
    licencias = [texto_sin_etiquetas(x) for x in licencias]
    autor = re.search(r'<td[^>]*id="fileinfotpl_aut"[^>]*>.*?</td>\s*<td[^>]*>(.*?)</td>', html, re.S)
    originales = []
    for enlace in pagina.enlaces:
        u = urlsplit(enlace)
        if u.netloc == "upload.wikimedia.org" and "/thumb/" not in u.path and "/transcoded/" not in u.path:
            originales.append(urlunsplit((u.scheme or "https", u.netloc, u.path, "", "")))
    if not originales or not licencias or not autor:
        raise ValueError("No se pudo verificar original, autor y licencia en la ficha.")
    return {"url_original": originales[0], "licencias": licencias, "autor": texto_sin_etiquetas(autor.group(1))}

def recolectar_entrada(entrada):
    carpeta = RAW / ("documentos" if entrada["modalidad"] == "documento" else "audio")
    carpeta.mkdir(parents=True, exist_ok=True)
    ficha = carpeta / (entrada["id"] + ".fuente.html")
    registro = carpeta / (entrada["id"] + ".json")
    url = "https://commons.wikimedia.org/wiki/File:" + quote(entrada["titulo"].replace(" ", "_"))
    if not ficha.exists():
        ficha.write_bytes(obtener(url))
    meta = metadatos_commons(ficha.read_text(encoding="utf-8"))
    if entrada["licencia_esperada"] not in meta["licencias"]:
        raise ValueError(f"Revisar licencia de {entrada['id']}: {meta['licencias']}")
    extension = Path(urlsplit(meta["url_original"]).path).suffix.lower()
    archivo = carpeta / (entrada["id"] + extension)
    if not archivo.exists():
        archivo.write_bytes(obtener(meta["url_original"]))
    contenido = archivo.read_bytes()
    fecha = json.loads(registro.read_text(encoding="utf-8"))["fecha_descarga"] if registro.exists() else datetime.now(timezone(timedelta(hours=-5))).isoformat(timespec="seconds")
    fila = dict(entrada, fuente_url=url, licencia=entrada["licencia_esperada"],
                acceso=meta["url_original"], autor=meta["autor"],
                fecha_descarga=fecha, archivo=archivo.relative_to(RAIZ).as_posix(),
                bytes=len(contenido), sha256=hashlib.sha256(contenido).hexdigest())
    registro.write_text(json.dumps(fila, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    ruta_robots = RAW / "robots.json"
    anteriores = json.loads(ruta_robots.read_text(encoding="utf-8")) if ruta_robots.exists() else []
    registros_robots = {x["url"]: x for x in anteriores}
    registros_robots.update({x[1]["url"]: x[1] for x in robots_por_host.values()})
    ruta_robots.write_text(json.dumps(list(registros_robots.values()), ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    return fila


In [3]:
CANDIDATOS = [
    {
        "id": "doc_001",
        "modalidad": "documento",
        "tipo": "impreso",
        "titulo": "Acta de Creación de los Símbolos para el Distrito de Mache.jpg",
        "licencia_esperada": "CC BY 4.0"
    },
    {
        "id": "doc_002",
        "modalidad": "documento",
        "tipo": "foto",
        "titulo": "Placa Calle Capon.jpg",
        "licencia_esperada": "CC BY 3.0"
    },
    {
        "id": "doc_003",
        "modalidad": "documento",
        "tipo": "manuscrito",
        "titulo": "Tarjeta de Ricardo Palma a Luis Antonio Eguiguren Escudero 24 octubre 1912.png",
        "licencia_esperada": "Public domain"
    },
    {
        "id": "clip_001",
        "modalidad": "audio",
        "tipo": "entrevista",
        "titulo": "Entrevista 2 en la Av. Rivagüero - El Agustino.ogg",
        "licencia_esperada": "CC BY-SA 4.0"
    }
]

registro_piloto = []
for entrada in CANDIDATOS:
    fila = recolectar_entrada(entrada)
    registro_piloto.append(fila)
    print(f"{fila['id']} | {fila['tipo']} | {fila['licencia']} | {fila['bytes']:,} bytes", flush=True)


doc_001 | impreso | CC BY 4.0 | 450,587 bytes
doc_002 | foto | CC BY 3.0 | 2,039,144 bytes
doc_003 | manuscrito | Public domain | 8,189,076 bytes
clip_001 | entrevista | CC BY-SA 4.0 | 1,663,962 bytes


### 1.2. Revisión del piloto y registro de fuentes

La inspección visual distingue descarga de aceptación: la placa se descarta por tener pocas palabras y no ser una fotografía de un documento en papel. El manuscrito incluye una transcripción impresa inferior, que debe quedar fuera del recorte usado en OCR y CER. El audio conserva su duración original; la presencia y los turnos de voz se confirmarán mediante escucha. `fuentes.csv` conserva también la procedencia del archivo descartado para documentar la decisión. Los CSV de resultados seguirán sin filas hasta procesar y revisar sus textos.


In [4]:
import struct

def duracion_ogg_vorbis(ruta):
    contenido = ruta.read_bytes()
    marca = contenido.find(b"\x01vorbis")
    if marca < 0:
        raise ValueError("Esta comprobación inicial solo admite Ogg Vorbis.")
    frecuencia = struct.unpack_from("<I", contenido, marca + 12)[0]
    posicion, granulos, flujos = 0, [], set()
    while posicion < len(contenido):
        if contenido[posicion:posicion + 4] != b"OggS":
            raise ValueError("Archivo Ogg incompleto o inválido.")
        segmentos = contenido[posicion + 26]
        longitud = sum(contenido[posicion + 27:posicion + 27 + segmentos])
        granulo = struct.unpack_from("<Q", contenido, posicion + 6)[0]
        if granulo != 2**64 - 1:
            granulos.append(granulo)
        flujos.add(struct.unpack_from("<I", contenido, posicion + 14)[0])
        posicion += 27 + segmentos + longitud
    if len(flujos) != 1:
        raise ValueError("Revisar archivo con varios flujos Ogg.")
    return max(granulos) / frecuencia

revision_piloto = {
    "doc_001": ("candidato", "Texto impreso legible; contiene nombres que deben protegerse antes de publicar."),
    "doc_002": ("descartado", "Pocas palabras y soporte de placa; se priorizan fotografías de documentos."),
    "doc_003": ("requiere_recorte", "La franja inferior contiene una transcripción impresa: excluirla antes de OCR y evaluación."),
    "clip_001": ("requiere_escucha", "Confirmar voz, condiciones y turnos mediante escucha; sin transcripción ni conteo de hablantes todavía.")
}
for fila in registro_piloto:
    fila["seleccion"], fila["observaciones"] = revision_piloto[fila["id"]]
    if fila["modalidad"] == "audio":
        fila["duracion_seg"] = duracion_ogg_vorbis(RAIZ / fila["archivo"])
    print(f"{fila['id']} | {fila['seleccion']} | {fila.get('duracion_seg', '')}")


doc_001 | candidato | 
doc_002 | descartado | 
doc_003 | requiere_recorte | 
clip_001 | requiere_escucha | 189.01043083900225


In [5]:
columnas_fuentes = ["id", "modalidad", "fuente_url", "licencia", "acceso",
                    "autor", "fecha_descarga", "archivo", "sha256", "seleccion", "observaciones"]
ruta_fuentes = RAIZ / "datos" / "fuentes.csv"
existentes = {}
if ruta_fuentes.exists():
    with ruta_fuentes.open(encoding="utf-8", newline="") as archivo:
        existentes = {fila["id"]: fila for fila in csv.DictReader(archivo)}
for fila in registro_piloto:
    existentes[fila["id"]] = {columna: fila.get(columna, "") for columna in columnas_fuentes}
with ruta_fuentes.open("w", encoding="utf-8", newline="") as archivo:
    escritor = csv.DictWriter(archivo, fieldnames=columnas_fuentes, lineterminator="\n")
    escritor.writeheader()
    escritor.writerows(existentes.values())
print("Fuentes documentadas:", len(existentes))
print("Los corpus de documentos y transcripciones se exportarán después de su procesamiento y revisión.")


Fuentes documentadas: 4
Los corpus de documentos y transcripciones se exportarán después de su procesamiento y revisión.


## 2. OCR con PaddleOCR

Documentar idioma y preprocesamiento. Conservar originales, respaldos crudos y recortes en `datos/raw/documentos/`, excluidos de Git. No rellenar el corpus con texto sintético.


## 3. ASR, VAD y diarización

Transcribir con faster-whisper y VAD, mediante `voz.py` o equivalente. Aplicar diarización al menos a un audio multihablante. Conservar originales y salidas pesadas en `datos/raw/audio/`, excluidos de Git. Exportar CSV, SRT y TXT; revisar manualmente una muestra para detectar alucinaciones en silencio, ruido o música.


## 4. Seudonimización y revisión humana

Revisar textos OCR y ASR antes de publicarlos. Sustituir nombres, DNI y otros datos personales reales cuando corresponda; documentar el procedimiento y contar las entidades reemplazadas. Revisar también metadatos e imágenes o audios que puedan publicarse.


## 5. Limpieza y descartes

Eliminar duplicados por hash de archivo. Reportar cantidades y motivos de descarte de documentos ilegibles y audios sin voz; reemplazarlos hasta cumplir los mínimos.


## 6. Análisis exploratorio con Plotly Express

Preparar tres gráficos interactivos: distribución por tipo de documento; duración por clip o minutos por fuente; distribución de longitud de los textos (caracteres OCR y palabras ASR, mediante paneles si hace falta). Redactar títulos que expresen conclusiones sustentadas en los datos. Conservar salidas visibles en GitHub.


## 7. Validación de extremos

Mostrar los tres documentos con texto OCR más largo y los tres con texto más corto; además, los tres clips de mayor y los tres de menor duración. Verificar coherencia con las distribuciones.


## 8. Exportación y comprobación de corpus

Guardar `datos/documentos.csv`, `datos/transcripciones.csv` y `datos/fuentes.csv` con los encabezados requeridos. Mostrar dimensiones y primeras cinco filas, comprobar identificadores y rutas, volúmenes, tres tipos y audio multihablante. `texto_referencia` se completará manualmente solo en las muestras de evaluación de la Parte 2.
